#### Data Loader

In [1]:
%pip install pandas

   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   ----------------------------------- ---- 9.4/10.8 MB 49.1 MB/s eta 0:00:01
   ---------------------------------------- 10.8/10.8 MB 44.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

#### (1)데이터 준비

In [8]:
path = 'https://raw.githubusercontent.com/DA4BAM/dataset/master/advertising.csv'
data = pd.read_csv(path)
print(data.head())

data_input = data[:10]
target = 'Sales'
x = data_input.drop(target, axis=1)
y = data_input.loc[:, target]
print(x.shape, y.shape)
print(y)

      TV  Radio  Newspaper  Sales
0  230.1   37.8       69.2   22.1
1   44.5   39.3       45.1   10.4
2   17.2   45.9       69.3    9.3
3  151.5   41.3       58.5   18.5
4  180.8   10.8       58.4   12.9
(10, 3) (10,)
0    22.1
1    10.4
2     9.3
3    18.5
4    12.9
5     7.2
6    11.8
7    13.2
8     4.8
9    10.6
Name: Sales, dtype: float64


#### (2) tensor로 변환

In [12]:
print(type(x))
print(type(y))

#numpy 변화
x_np = x.values
y_np = y.values

print(type(x_np), type(y_np))
print(x_np.dtype, y_np.dtype)
print(x_np.shape, y_np.shape)

y_np = y_np.reshape(10,1)
print(x_np.shape, y_np.shape)

<class 'pandas.DataFrame'>
<class 'pandas.Series'>
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
float64 float64
(10, 3) (10,)
(10, 3) (10, 1)


In [13]:
# 텐서로 변경
# dtype : float64 -> torch는 float32를 기본적인 가중치로 사용

x_tensor = torch.tensor(x_np, dtype=torch.float32)
y_tensor = torch.tensor(y_np, dtype=torch.float32)

print(x_tensor.shape, y_tensor.shape)

torch.Size([10, 3]) torch.Size([10, 1])


In [14]:
x_tensor[:5]

tensor([[230.1000,  37.8000,  69.2000],
        [ 44.5000,  39.3000,  45.1000],
        [ 17.2000,  45.9000,  69.3000],
        [151.5000,  41.3000,  58.5000],
        [180.8000,  10.8000,  58.4000]])

In [15]:
y_tensor[:5]

tensor([[22.1000],
        [10.4000],
        [ 9.3000],
        [18.5000],
        [12.9000]])

#### (3) Data Loader 준비
- Data Loader는 텐서 데이터셋을 입력으로 받음
    - 그래서, 텐서 x,y를 하나의 텐서 데이터 셋으로 합치기
    - 그 다음 Data Loader 생성

#### 1) 텐서 데이터셋

In [20]:
dataset = TensorDataset(x_tensor, y_tensor)

dataset[:5] #5개 행만 조회

dataset.tensors # 전체 조회

dataset.tensors[0] # x조회

dataset.tensors[1] # y조회

len(dataset) #데이터 건수

10

#### 2) 데이터 로더

In [22]:
data_loader = DataLoader(dataset,batch_size=3, shuffle=True)

batch_size = : 한 번에 모델에 전달할 데이터의 개수 지정 <br>
shuffle = True :  학습 데이터를 매 에포크 시작시 무작위로 섞을지를 지정 <br>
&emsp;&emsp; True : 데이터를 무작위로 섞음<br>
&emsp;&emsp; False : 데이터의 원래 순서를 유지함<br><br>

에포크(Epoch) : 전체 학습 데이터를 모델이 한번 모두 학습하는 것<br>
&emsp; 예를 들어 데이터가 8개이고 batch_size=3이라면,<br>
&emsp;&emsp; Batch 1 : 데이터 3개 학습<br>
&emsp;&emsp; Batch 2 : 데이터 3개 학습<br>
&emsp;&emsp; Batch 3 : 데이터 2개 학습<br>
&emsp; 이렇게 3개의 배치를 모두 학습하면 1Epoch 완료<br><br>

- 데이터 로더로부터 데이터 추출하기

In [23]:
for (x_b, y_b) in data_loader:
    print(x_b)
    print(y_b)
    break

tensor([[  8.6000,   2.1000,   1.0000],
        [180.8000,  10.8000,  58.4000],
        [151.5000,  41.3000,  58.5000]])
tensor([[ 4.8000],
        [12.9000],
        [18.5000]])


In [ ]:
#데이터 로더로부터 배치 단위로 데이터 로딩
for batch_id, (x_b, y_b) in enumerate(data_loader):
    print(f'batch_id : {batch_id}')
    print(x_b)
    print(y_b)
    print('-'*50)

batch_id :0
tensor([[ 8.6000,  2.1000,  1.0000],
        [57.5000, 32.8000, 23.5000],
        [44.5000, 39.3000, 45.1000]])
tensor([[ 4.8000],
        [11.8000],
        [10.4000]])
--------------------------------------------------
batch_id :1
tensor([[ 17.2000,  45.9000,  69.3000],
        [151.5000,  41.3000,  58.5000],
        [199.8000,   2.6000,  21.2000]])
tensor([[ 9.3000],
        [18.5000],
        [10.6000]])
--------------------------------------------------
batch_id :2
tensor([[180.8000,  10.8000,  58.4000],
        [  8.7000,  48.9000,  75.0000],
        [230.1000,  37.8000,  69.2000]])
tensor([[12.9000],
        [ 7.2000],
        [22.1000]])
--------------------------------------------------
batch_id :3
tensor([[120.2000,  19.6000,  11.6000]])
tensor([[13.2000]])
--------------------------------------------------


In [26]:
print(f'데이터로더의 데이터 셋 : {len(data_loader.dataset)}')
print(f'데이터로더의 배치 수 : {len(data_loader)}')

데이터로더의 데이터 셋 : 10
데이터로더의 배치 수 : 4
